In [1]:
import numpy as np
from pendulum import Pendulum
# from ASNN import ReplayBuffer_ASNN


In [2]:
## [test simulation] ##
sim_step = 100
delta_t = 0.05
pendulum = Pendulum()
for i in range(sim_step):
    pendulum.update(u=[2.0 * np.sin(i/5.0)], delta_t=delta_t) # u is the control input to the pendulum, [ torque[Nm] ]
pendulum.show_animation(interval_ms=delta_t*1000) # show animation

In [ ]:
# simulation settings
delta_t = 0.05 # [sec]
sim_steps = 150 # [steps]
print(f"[INFO] delta_t : {delta_t:.2f}[s] , sim_steps : {sim_steps}[steps], total_sim_time : {delta_t*sim_steps:.2f}[s]")

# initialize a pendulum as a control target
pendulum = Pendulum(
    mass_of_pole = 1.0,
    length_of_pole = 1.0,
    max_torque_abs = 2.0,
    max_speed_abs = 8.0,
    delta_t = delta_t,
    visualize = True,
)
pendulum.reset(
    init_state = np.array([np.pi, 0.0]), # [theta(rad), theta_dot(rad/s)]
)

# initialize a mppi controller for the pendulum
mppi = MPPIControllerForPendulum(
    delta_t = delta_t,
    mass_of_pole = 1.0,
    length_of_pole = 1.0,
    max_torque_abs = 2.0,
    max_speed_abs = 8.0,
    horizon_step_T = 20,
    number_of_samples_K = 2000,
    param_exploration = 0.05,
    param_lambda = 0.5,
    param_alpha = 0.8,
    sigma = 1.0,
    stage_cost_weight    = np.array([1.0, 0.1]), # weight for [theta, theta_dot]
    terminal_cost_weight = 5.0 * np.array([1.0, 0.1]), # weight for [theta, theta_dot]
)

# simulation loop
for i in range(sim_steps):

    # get current state of pendulum
    current_state = pendulum.get_state()

    # calculate input force with MPPI
    input_torque, input_torque_sequence = mppi.calc_control_input(
        observed_x = current_state
    )

    # print current state and input torque
    print(f"Time: {i*delta_t:>2.2f}[s], theta={current_state[0]:>+3.3f}[rad], theta_dot={current_state[1]:>+3.3f}[rad/s], input torque={input_torque:>+3.2f}[Nm]", end="")
    print(", # currently staying upright #" if abs(current_state[0]) < 0.1 and abs(current_state[1] < 0.1) else "")

    # update states of pendulum
    pendulum.update(u=[input_torque], delta_t=delta_t)

# show animation
pendulum.show_animation(interval_ms=int(delta_t * 1000))
# save animation
pendulum.save_animation("mppi_pendulum.mp4", interval=int(delta_t * 1000), movie_writer="ffmpeg") # ffmpeg is required to write mp4 file